In [1]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers_sae import _autoreload

# from transformers_sae import _autoreload
from transformers_sae.ops import MemoryTrackingMode
from transformers_sae.replacement_model import GemmaReplacement, make_replacement_model

# Tweak TRAINING_BATCH_SIZE for your hardware if necessary
if torch.cuda.is_available():
    TRAINING_DEVICE = "cuda:0"
    TRAINING_BATCH_SIZE = 2
elif torch.mps.is_available():
    TRAINING_DEVICE = "mps:0"
    TRAINING_BATCH_SIZE = 2
else:
    TRAINING_DEVICE = "cpu"
    TRAINING_BATCH_SIZE = 2

model_id = "google/gemma-2-2b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
training_dataset = load_dataset(
    "monology/pile-uncopyrighted-parquet",
    split="train",
    streaming=True,
    columns=["text"],
)
validation_dataset = load_dataset(
    "monology/pile-test-val",
    split="validation",
    revision="refs/convert/parquet",
    streaming=True,
    columns=["text"],
)

with MemoryTrackingMode() as mtm:
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map=TRAINING_DEVICE,
        dtype=torch.bfloat16,
        use_safetensors=True,
    )
    model = make_replacement_model(
        model,
        {},
        num_layers=model.config.num_hidden_layers,
        context_length=1024,  # model.config.max_position_embeddings,
        d_model=model.config.hidden_size,
        layer_path="model.layers",
        replacement_class=GemmaReplacement,
    )
    model.eval()
    model.requires_grad_(False)

print(model)
print(mtm.memory_max)
print(mtm.memory_cur)

/cloud-dev/.venv/lib/python3.13/site-packages/codefind/registry.py:46: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, types.FunctionType):


Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1987 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

GemmaReplacementInstance(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemm

In [52]:
import os

from transformers_sae.ops import method_to_saes

CHECKPOINT_BASE_PATH = f"{os.getenv('HF_BUCKET_LOCAL')}/gemma_2_2b"

sae_layer = 0
distill_layer = 1

# sae_layer = model.num_layers - 2
# distill_layer = model.num_layers - 1

saes = method_to_saes(
    CHECKPOINT_BASE_PATH,
    "gemma_scope_canonical_l0",
    [sae_layer],
    TRAINING_DEVICE,
)
sae = saes[sae_layer]
sae.eval()
sae.onload()

Loaded gemma scope 0 with target L0=105


In [53]:
from transformers_sae.activation_data import TrainingBatch


@torch.no_grad()
@torch.autocast(
    device_type="cuda" if model.device.type == "cuda" else "cpu",
    dtype=torch.bfloat16,
)
def run_model_and_sae(batch: TrainingBatch):
    base_model_output = make_activation_batch(
        model, [(sae_layer, "layer"), (distill_layer, "layer")], batch, end_layer=distill_layer+1
    )
    sae_output = sae(
        base_model_output[sae_layer].layer_output,
        token_mask=batch.token_mask,
    )
    return (
        base_model_output[sae_layer].layer_output,
        base_model_output[distill_layer].layer_output,
        base_model_output[sae_layer].layer_output - sae_output,
    )


@torch.autocast(
    device_type="cuda" if model.device.type == "cuda" else "cpu",
    dtype=torch.bfloat16,
)
def get_distilled_layer_output(
    training_layer: torch.nn.Module, batch: TrainingBatch, layer_input: torch.Tensor
):
    input_args, input_kwargs = model.get_base_model_args(batch, layer_input, False)
    layer_args, layer_kwargs = model.get_layer_args(
        distill_layer,
        training_layer,
        *input_args,
        **input_kwargs,
        use_cache=False,
    )
    return training_layer(*layer_args, **layer_kwargs)


In [65]:
model.config

Gemma2Config {
  "architectures": [
    "Gemma2ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "attn_logit_softcapping": 50.0,
  "bos_token_id": 2,
  "cache_implementation": "hybrid",
  "dtype": "bfloat16",
  "eos_token_id": 1,
  "final_logit_softcapping": 30.0,
  "head_dim": 256,
  "hidden_act": "gelu_pytorch_tanh",
  "hidden_activation": "gelu_pytorch_tanh",
  "hidden_size": 2304,
  "initializer_range": 0.02,
  "intermediate_size": 9216,
  "layer_types": [
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "full_attention",
    "sliding_attention",
    "full_atten

In [ ]:
from transformers_sae.tokenization import make_dataloader
from transformers_sae.activation_data import make_activation_batch
from transformers.models.gemma2.modeling_gemma2 import Gemma2DecoderLayer
from tqdm.auto import tqdm

NUM_TRAINING_TOKENS = int(1e8)

distilled_layer = Gemma2DecoderLayer(model.config, distill_layer)
distilled_layer.load_state_dict(model.get_layer(distill_layer).state_dict())
distilled_layer.to(TRAINING_DEVICE)
distilled_layer.to(torch.float32)
distilled_layer.train()

optimizer = torch.optim.Adam(distilled_layer.parameters(), lr=1e-4)
progress = tqdm(
    total=NUM_TRAINING_TOKENS, desc="Distilling final layer from error term"
)
num_used_tokens = 0
for batch in make_dataloader(
    model,
    tokenizer,
    training_dataset,
    max_tokens=NUM_TRAINING_TOKENS,
    tokenizer_batch_size=256,
    inference_batch_size=TRAINING_BATCH_SIZE,
):
    batch.to(model.device)
    optimizer.zero_grad()

    sae_layer_output, distill_layer_output, error_term = run_model_and_sae(batch)

    # compare to gaussian noise with same stdmean as input?

    with torch.autocast(
        device_type="cuda" if model.device.type == "cuda" else "cpu",
        dtype=torch.bfloat16,
    ):
        actual_output = get_distilled_layer_output(distilled_layer, batch, error_term)
        # actual_output = get_distilled_layer_output(
        #     distilled_layer, batch, sae_layer_output
        # )
        loss = ((actual_output - distill_layer_output) ** 2).sum(dim=-1).mean()
        rre = (
            torch.linalg.vector_norm(
                actual_output - distill_layer_output, dim=-1, dtype=torch.float32
            )
            / (
                torch.linalg.vector_norm(
                    distill_layer_output, dim=-1, dtype=torch.float32
                )
                + 1e-8
            )
        ).mean()
    loss.backward()
    optimizer.step()

    num_used_tokens += batch.num_tokens
    progress.total = max(NUM_TRAINING_TOKENS, num_used_tokens)
    progress.set_postfix({"rre": rre.item(), "loss": loss.item()}, refresh=False)
    progress.update(batch.num_tokens)

progress.close()

Distilling final layer from error term:   0%|          | 0/100000000 [00:00<?, ?it/s]

In [ ]:
import numpy as np

NUM_VALIDATION_TOKENS = int(1e6)
num_used_tokens = 0
progress = tqdm(total=NUM_VALIDATION_TOKENS, desc="Evaluating")
validation_kl = []
validation_rre = []


def gmean(values):
    return np.exp(
        np.mean(np.log(np.clip(np.array(values), a_min=1e-9, a_max=None)))
    ).item()


distilled_layer.eval()
for batch in make_dataloader(
    model,
    tokenizer,
    training_dataset,
    max_tokens=NUM_VALIDATION_TOKENS,
    tokenizer_batch_size=256,
    inference_batch_size=TRAINING_BATCH_SIZE,
):
    batch.to(model.device)

    sae_layer_output, distill_layer_output, error_term = run_model_and_sae(batch)
    with (
        torch.autocast(
            device_type="cuda" if model.device.type == "cuda" else "cpu",
            dtype=torch.bfloat16,
        ),
        torch.no_grad(),
    ):
        actual_output = get_distilled_layer_output(distilled_layer, batch, error_term)
        # actual_output = get_distilled_layer_output(
        #     distilled_layer, batch, sae_layer_output
        # )
        rre = torch.linalg.vector_norm(
            actual_output - distill_layer_output, dim=-1, dtype=torch.float32
        ) / (
            torch.linalg.vector_norm(distill_layer_output, dim=-1, dtype=torch.float32)
            + 1e-8
        )
        # kl_loss = torch.nn.functional.kl_div(
        #     model.get_logits(actual_output).log_softmax(-1),
        #     model.get_logits(distill_layer_output).log_softmax(-1),
        #     log_target=True,
        #     reduction="none",
        # ).sum(dim=-1)
        # validation_kl = np.concat([validation_kl, kl_loss.cpu().flatten().numpy()])
        validation_rre = np.concat([validation_rre, rre.cpu().flatten().numpy()])

    num_used_tokens += batch.num_tokens
    progress.total = max(NUM_VALIDATION_TOKENS, num_used_tokens)
    progress.set_postfix(
        {"rre": np.mean(validation_rre), "gmean_kl": gmean(validation_kl)},
        refresh=False,
    )
    progress.update(batch.num_tokens)

progress.close()

Evaluating:   0%|          | 0/1000000 [00:00<?, ?it/s]

/cloud-dev/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/cloud-dev/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [56]:
from safetensors.torch import save_model

os.makedirs(f"{os.getenv('HF_BUCKET_LOCAL')}/error_distillation/", exist_ok=True)
save_model(distilled_layer, f"{os.getenv('HF_BUCKET_LOCAL')}/error_distillation/layer_{distill_layer}_from_error.safetensors")